# Baseline Experiments for RETFound + Q2L

4 baselines to isolate the Q2L decoder contribution. All reuse the **identical** data pipeline, loss, optimizer, scheduler, augmentations, and evaluation as Exp C (Joint CFP+UWF). Only the model head and backbone freezing differ.

| ID | Backbone | Head | Rationale |
|----|----------|------|-----------|
| **B1** | Fine-tuned | GAP → Linear(7) | Simplest baseline, same backbone training |
| **B2** | Fine-tuned | GAP → MLP(1024→512→7) | More expressive head without cross-attention |
| **B3** | Frozen (all 24 blocks) | GAP → Linear(7) | Fine-tuning impact with simple head |
| **B4** | Frozen (all 24 blocks) | Q2L Decoder | Q2L contribution on frozen features |

**Key comparisons:** B1 vs Main → Q2L decoder value · B3 vs B1 → fine-tuning impact · B3 vs B4 → Q2L on frozen features

In [1]:
import gc
import json
import os
import sys
import time
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

import numpy as np
import torch
import torch.nn as nn

# ── Kaggle paths (robust discovery — do NOT hard-code exact mount depth) ──
# Kaggle mounts an uploaded dataset under /kaggle/input/<dataset-slug>/...,
# but the slug can differ from the dataset's display title, and a
# folder-uploaded dataset is often mounted ONE LEVEL DEEPER than expected
# (/kaggle/input/<slug>/<original-folder-name>/<file>). Hard-coding the
# full path silently breaks whenever that nesting doesn't match — which is
# exactly what happened here. Search for the files by name instead, the
# same pattern already used successfully in multimodel.ipynb for the
# ResNet-50 weights.
INPUT_ROOT   = Path("/kaggle/input")
WORKING_ROOT = "/kaggle/working"

def _find_first(pattern, base=INPUT_ROOT):
    """Recursively find the first file matching `pattern` under `base`."""
    return next(base.rglob(pattern), None)

# RETFound weights: search recursively for the checkpoint by filename
# instead of assuming a fixed dataset-folder depth.
_weights_hit = _find_first("RETFound_mae_meh.pth")
WEIGHTS_PATH = str(_weights_hit) if _weights_hit is not None else \
    "/kaggle/input/retfound-meh-weights/RETFound_mae_meh.pth"  # fallback for error message only

# Project code: locate the directory that actually contains train.py /
# src/, rather than assuming it sits directly at /kaggle/input/q2l-mmrdr-code.
_train_py_hit = _find_first("train.py")
CODE_ROOT = str(_train_py_hit.parent) if _train_py_hit is not None else \
    "/kaggle/input/q2l-mmrdr-code"  # fallback for error message only

sys.path.insert(0, CODE_ROOT)

print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}")
print(f"Code:    {CODE_ROOT}"    + ("" if _train_py_hit is not None else "  (NOT FOUND — see validation cell)"))
print(f"Weights: {WEIGHTS_PATH}" + ("" if _weights_hit  is not None else "  (NOT FOUND — see validation cell)"))
print(f"Working: {WORKING_ROOT}")


PyTorch 2.10.0+cu128 | CUDA True
Code:    /kaggle/input/q2l-mmrdr-code
Weights: /kaggle/input/datasets/sanchitgarg999/retfound-meh-weights/RETFound_mae_meh.pth
Working: /kaggle/working


In [2]:
def extract_mmrdr():
    """Reconstruct and extract the MMRDR split archive if not already done."""
    target = Path(WORKING_ROOT)
    cfp_path = target / "dataset" / "MMRDR-CFP"
    uwf_path = target / "dataset" / "MMRDR-UWF"

    if cfp_path.exists() and uwf_path.exists():
        print("MMRDR dataset already extracted, skipping.")
        return

    import shutil
    import subprocess

    parts = sorted(Path("/kaggle/input").rglob("MMRDR.zip.*"),
                   key=lambda p: p.name)
    if not parts:
        raise FileNotFoundError(
            "No MMRDR.zip.* found under /kaggle/input/. "
            "Attach Kaggle datasets MMRDR001 through MMRDR009.")
    print(f"Found {len(parts)} archive parts.")

    # Symlink all parts into one directory so 7z can discover them
    tmp = target / "_mmrdr_parts"
    tmp.mkdir(exist_ok=True)
    for p in parts:
        link = tmp / p.name
        if not link.exists():
            link.symlink_to(p)

    print("Extracting (this may take several minutes)...")
    subprocess.run(
        ["7z", "x", str(tmp / parts[0].name), f"-o{target}", "-y"],
        check=True, stdout=subprocess.DEVNULL)
    shutil.rmtree(tmp)

    # Handle zip structures without a dataset/ prefix
    if not cfp_path.exists():
        if (target / "MMRDR-CFP").exists():
            (target / "dataset").mkdir(exist_ok=True)
            (target / "MMRDR-CFP").rename(cfp_path)
            if (target / "MMRDR-UWF").exists():
                (target / "MMRDR-UWF").rename(uwf_path)
        elif (target / "MMRDR" / "MMRDR-CFP").exists():
            (target / "dataset").mkdir(exist_ok=True)
            (target / "MMRDR" / "MMRDR-CFP").rename(cfp_path)
            if (target / "MMRDR" / "MMRDR-UWF").exists():
                (target / "MMRDR" / "MMRDR-UWF").rename(uwf_path)

    if not cfp_path.exists():
        dirs = [p.name for p in target.iterdir() if p.is_dir()]
        raise FileNotFoundError(
            f"Expected {cfp_path} after extraction but not found. "
            f"Directories in {target}: {dirs}")

    print("MMRDR extraction complete.")

extract_mmrdr()

Found 9 archive parts.
Extracting (this may take several minutes)...
MMRDR extraction complete.


In [3]:
# ── Data root discovery (fixes B1–B4 FileNotFoundError) ──────────────────
# ROOT CAUSE: extract_mmrdr() (above) places the dataset at
#   WORKING_ROOT/dataset/MMRDR-CFP  and  WORKING_ROOT/dataset/MMRDR-UWF
# but src/config.py's Config.project_root is the directory that must
# DIRECTLY contain MMRDR-CFP/ and MMRDR-UWF/ (src/dataset.py's
# build_dataloaders() does `Path(cfg.project_root) / cfg.cfp_dir`, and
# cfg.cfp_dir defaults to the bare folder name "MMRDR-CFP", with no
# "dataset/" prefix). run_baseline() was passing project_root=WORKING_ROOT
# (i.e. "/kaggle/working") — one directory level too shallow — so
# Config resolved CFP's CSV to /kaggle/working/MMRDR-CFP/FP.csv, which
# never existed; the real file is one level deeper, under .../dataset/.
#
# Rather than hard-code "dataset/" as the missing segment (which would
# just move the same class of bug one level down if extract_mmrdr()'s
# internal layout ever changes again), we discover the correct
# project_root directly from where the annotation CSVs actually are.
def _discover_data_root():
    """Return the directory that directly contains MMRDR-CFP/ and
    MMRDR-UWF/ (the value src/config.py's Config.project_root must be),
    found by locating FP.csv itself rather than assuming a fixed
    subfolder name.
    """
    for base in (Path(WORKING_ROOT), INPUT_ROOT):
        for cfp_csv_hit in base.rglob("FP.csv"):
            cfp_dir = cfp_csv_hit.parent            # .../MMRDR-CFP
            if cfp_dir.name != "MMRDR-CFP":
                continue
            candidate_root = cfp_dir.parent          # folder containing MMRDR-CFP
            if (candidate_root / "MMRDR-UWF" / "UWF.csv").exists():
                return candidate_root
    return None

DATA_ROOT = _discover_data_root()
print(f"Data root: {DATA_ROOT}" if DATA_ROOT is not None
      else "Data root: NOT FOUND (see validation cell below)")


Data root: /kaggle/working/dataset


In [4]:
from src.config import Config, LESION_NAMES, NUM_CLASSES
from src.dataset import build_dataloaders, count_class_positives_fast
from src.backbone import RETFoundBackbone
from src.model import Q2LLesionModel
from src.losses import AsymmetricLoss, compute_effective_number_weights
from src.metrics import compute_metrics
from src.threshold import run_threshold_optimization, _collect_predictions
from src.utils import (
    set_seed, setup_logging, save_checkpoint, load_checkpoint,
    ModelEMA, Timer, append_epoch_history,
)
from train import CosineWarmupScheduler, train_one_epoch, validate

print("All project imports OK")

All project imports OK


In [5]:
class BaselineModel(nn.Module):
    """RETFound backbone + Global Average Pooling + Linear/MLP head.

    Drop-in replacement for Q2LLesionModel: same forward signature,
    same get_param_groups() / param_summary() interface so the
    existing training loop works unmodified.
    """

    def __init__(self, cfg, weights_path=None, head_type="linear",
                 freeze_all=False):
        super().__init__()
        self.cfg = cfg
        self.backbone = RETFoundBackbone(
            image_size=cfg.image_size,
            patch_size=cfg.patch_size,
            pretrained=cfg.backbone_pretrained,
            freeze_blocks=list(range(24)) if freeze_all else cfg.freeze_blocks,
            weights_path=weights_path,
            gradient_checkpointing=(not freeze_all) and cfg.gradient_checkpointing,
        )
        d = self.backbone.OUT_CHANNELS  # 1024
        self.pool = nn.AdaptiveAvgPool2d(1)
        if head_type == "mlp":
            self.head = nn.Sequential(
                nn.Linear(d, 512), nn.GELU(), nn.Dropout(0.1),
                nn.Linear(512, cfg.num_labels),
            )
        else:
            self.head = nn.Linear(d, cfg.num_labels)

    def forward(self, images, modality_ids=None, return_queries=False):
        features = self.backbone(images)            # (B, 1024, Hf, Wf)
        logits = self.head(self.pool(features).flatten(1))  # (B, 7)
        return (logits, None) if return_queries else logits

    def get_param_groups(self):
        groups = []
        bb = list(self.backbone.backbone_params())  # only requires_grad=True
        if bb:
            groups.append({"params": bb, "lr": self.cfg.lr_backbone,
                           "name": "backbone"})
        groups.append({"params": list(self.head.parameters()),
                       "lr": self.cfg.lr_decoder, "name": "head"})
        return groups

    def param_summary(self):
        bt, br = self.backbone.num_total_params(), self.backbone.num_trainable_params()
        ht = sum(p.numel() for p in self.head.parameters())
        hr = sum(p.numel() for p in self.head.parameters() if p.requires_grad)
        return dict(backbone_total=bt, backbone_trainable=br,
                    decoder_total=ht, decoder_trainable=hr,
                    total=bt + ht, trainable=br + hr)

In [6]:
def run_baseline(baseline_id, head_type, freeze_all, description):
    """Train → threshold optimisation → test evaluation → cleanup."""
    experiment = f"baseline_{baseline_id}"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    set_seed(42)

    # Config (matches Exp C exactly, only head / freezing differs)
    cfg = Config(
        experiment=experiment,
        description=description,
        modalities=["cfp", "uwf"],
        balanced_sampling=True,
        use_modality_conditioning=False,
        project_root=str(DATA_ROOT),  # FIX: was WORKING_ROOT (one level too shallow — see data-root discovery cell)
        freeze_blocks=list(range(24)) if freeze_all else [],
        query_diversity_weight=0.0,
    )
    if freeze_all and baseline_id != "B4":
        cfg.gradient_checkpointing = False

    cfg.output_dir.mkdir(parents=True, exist_ok=True)
    logger = setup_logging(cfg.output_dir, name=experiment)
    timer = Timer()
    cfg.save()

    logger.info("=" * 70)
    logger.info(f"BASELINE {baseline_id}: {description}")
    logger.info(f"Output: {cfg.output_dir}")
    logger.info("=" * 70)

    # Data
    train_loader, val_loader, test_loaders = build_dataloaders(cfg)
    logger.info(f"Train: {len(train_loader)} batches | "
                f"Val: {len(val_loader)} batches")

    # Model
    if baseline_id == "B4":
        model = Q2LLesionModel(cfg, weights_path=WEIGHTS_PATH)
    else:
        model = BaselineModel(cfg, WEIGHTS_PATH, head_type, freeze_all)
    model = model.to(device)

    s = model.param_summary()
    logger.info(f"Backbone: {s['backbone_total']:,} "
                f"({s['backbone_trainable']:,} trainable)")
    logger.info(f"Head:     {s['decoder_total']:,} "
                f"({s['decoder_trainable']:,} trainable)")
    logger.info(f"Total:    {s['trainable']:,} trainable / {s['total']:,}")

    # Loss (class-balanced ASL, identical to main)
    class_weights = None
    if cfg.class_balance_enabled:
        ds = train_loader.dataset
        ds_list = ds.datasets if hasattr(ds, "datasets") else [ds]
        counts = count_class_positives_fast(ds_list, NUM_CLASSES)
        class_weights = compute_effective_number_weights(
            counts, cfg.class_balance_beta, NUM_CLASSES)
    criterion = AsymmetricLoss(
        gamma_pos=cfg.asl_gamma_pos, gamma_neg=cfg.asl_gamma_neg,
        clip=cfg.asl_clip, reduction=cfg.loss_reduction,
        class_weights=class_weights,
    ).to(device)

    # Optimizer / Scheduler / Scaler / EMA
    optimizer = torch.optim.AdamW(
        model.get_param_groups(), weight_decay=cfg.weight_decay)
    scheduler = CosineWarmupScheduler(optimizer, cfg.warmup_epochs, cfg.epochs)
    scaler = torch.amp.GradScaler(enabled=cfg.use_amp)
    ema = ModelEMA(model, decay=cfg.ema_decay)

    # Training loop
    best_val_f1 = 0.0
    patience_ctr = 0
    global_step = 0

    logger.info(f"\nTraining (max {cfg.epochs} ep, patience {cfg.patience})…")

    for epoch in range(1, cfg.epochs + 1):
        t0 = time.time()
        if hasattr(train_loader, "sampler") and \
                hasattr(train_loader.sampler, "set_epoch"):
            train_loader.sampler.set_epoch(epoch)

        train_loss, global_step = train_one_epoch(
            model, train_loader, optimizer, criterion, scaler, device, cfg,
            ema, None,
            global_step=global_step, epoch=epoch,
            best_val_f1=best_val_f1, logger=logger, scheduler=scheduler,
        )

        mid = cfg.output_dir / "mid_epoch.pth"
        if mid.exists():
            mid.unlink()

        use_ema = not (cfg.ema_warmup_epochs > 0
                       and epoch <= cfg.ema_warmup_epochs)
        val_model = ema.ema_model if use_ema else model
        val_metrics = validate(
            val_model, val_loader, device, cfg, criterion=criterion)
        val_f1 = val_metrics["macro_f1"]
        val_loss = val_metrics.get("val_loss", float("nan"))

        scheduler.step()
        dt = time.time() - t0

        saved = False
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_ctr = 0
            save_checkpoint(
                model, optimizer, scheduler, scaler, epoch, best_val_f1,
                cfg.output_dir / "best_model.pth",
                ema_model=ema.ema_model,
                inference_only=True, global_step=global_step)
            saved = True
        else:
            patience_ctr += 1

        save_checkpoint(
            model, optimizer, scheduler, scaler, epoch, best_val_f1,
            cfg.output_dir / "last_model.pth",
            ema_model=ema.ema_model,
            inference_only=False, global_step=global_step)

        per_cls = "  ".join(
            f"{n}={val_metrics.get(f'{n}_f1', 0):.3f}" for n in LESION_NAMES)
        logger.info(
            f"Ep {epoch:3d}/{cfg.epochs} | "
            f"loss={train_loss:.4f} val={val_loss:.4f} | "
            f"F1={val_f1:.4f} (best={best_val_f1:.4f}) | "
            f"{dt:.0f}s{'  \u2605' if saved else ''}")
        logger.info(f"  {per_cls}")

        lrs = scheduler.get_last_lr()
        append_epoch_history(cfg.output_dir, {
            "epoch": epoch,
            "train_loss": round(train_loss, 6),
            "val_loss": round(val_loss, 6) if val_loss == val_loss else None,
            "macro_f1": val_f1,
            "best_macro_f1": best_val_f1,
            **{f"{n}_f1": val_metrics.get(f"{n}_f1") for n in LESION_NAMES},
            "lr_backbone": lrs[0],
            "lr_decoder": lrs[-1],
            "epoch_time_s": round(dt, 1),
            "checkpoint_saved": saved,
        })

        if patience_ctr >= cfg.patience:
            logger.info(f"Early stopping at epoch {epoch}")
            break

    logger.info(f"\nTraining done. Best F1={best_val_f1:.4f} | "
                f"{timer.elapsed_str()}")

    # Post-training: load best → threshold opt → test eval
    best_path = cfg.output_dir / "best_model.pth"
    if not best_path.exists():
        best_path = cfg.output_dir / "last_model.pth"
    ckpt = load_checkpoint(best_path, model, device=device)
    if "ema_state_dict" in ckpt:
        model.load_state_dict(ckpt["ema_state_dict"])
        logger.info("Using EMA weights for evaluation")
    model.eval()

    logger.info("\n" + "=" * 70)
    logger.info("THRESHOLD OPTIMIZATION")
    logger.info("=" * 70)
    thresholds, _, _ = run_threshold_optimization(
        model, val_loader, cfg, device, logger)

    logger.info("\n" + "=" * 70)
    logger.info("TEST EVALUATION")
    logger.info("=" * 70)
    results = {}
    for mod, tl in test_loaders.items():
        probs, labels = _collect_predictions(model, tl, device, cfg.use_amp)
        m05 = compute_metrics(probs, labels, thresholds=None)
        mopt = compute_metrics(probs, labels, thresholds=thresholds)
        results[mod] = {
            "metrics_at_0.5": m05,
            "metrics_at_optimized": mopt,
            "thresholds": thresholds,
        }
        logger.info(
            f"  {mod.upper()}: F1@0.5={m05['macro_f1']:.4f} \u2192 "
            f"F1@opt={mopt['macro_f1']:.4f}  "
            f"(\u0394={mopt['macro_f1'] - m05['macro_f1']:+.4f})")

    with open(cfg.output_dir / "results.json", "w") as f:
        json.dump(results, f, indent=2, default=str)
    logger.info(f"\nResults saved to {cfg.output_dir / 'results.json'}")

    # GPU cleanup
    del model, optimizer, scheduler, scaler, ema, criterion
    del train_loader, val_loader, test_loaders
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()

    logger.info(f"Total time for {baseline_id}: {timer.elapsed_str()}")
    logger.info("=" * 70 + "\n")
    return results

In [7]:
def print_comparison():
    """Load every results.json and print a side-by-side Macro-F1 table."""
    out = Path(WORKING_ROOT) / "outputs"
    exps = [
        ("B3 (Frozen+Linear)",  "baseline_B3"),
        ("B4 (Frozen+Q2L)",     "baseline_B4"),
        ("B1 (FT+Linear)",      "baseline_B1"),
        ("B2 (FT+MLP)",         "baseline_B2"),
        ("Main (FT+Q2L)",       "exp_c_joint"),
    ]

    w = 78
    print(f"\n{'=' * w}")
    print("BASELINE COMPARISON \u2014 Macro F1 on Test Sets")
    print(f"{'=' * w}")
    hdr = (f"{'Experiment':<24}{'CFP @0.5':>10}{'CFP @opt':>10}"
           f"{'UWF @0.5':>10}{'UWF @opt':>10}")
    print(hdr)
    print("-" * w)

    def _get(r, mod, key):
        v = r.get(mod, {}).get(key, {}).get("macro_f1")
        return f"{v:.4f}" if isinstance(v, (int, float)) else "  \u2014"

    for label, name in exps:
        rp = out / name / "results.json"
        if not rp.exists():
            print(f"{label:<24}  (results not available)")
            continue
        r = json.loads(rp.read_text())
        c5  = _get(r, "cfp", "metrics_at_0.5")
        co  = _get(r, "cfp", "metrics_at_optimized")
        u5  = _get(r, "uwf", "metrics_at_0.5")
        uo  = _get(r, "uwf", "metrics_at_optimized")
        print(f"{label:<24}{c5:>10}{co:>10}{u5:>10}{uo:>10}")

    print(f"{'=' * w}\n")

In [8]:
# ── Pre-flight validation (Task 4): fail fast with clear, actionable errors ──
def _check(label, ok, detail_ok, detail_fail):
    mark = "✓" if ok else "✗"
    print(f"{mark} {label}: {detail_ok if ok else detail_fail}")
    return ok

_all_ok = True

# 1. RETFound weights found
_ok = _weights_hit is not None
_all_ok &= _check(
    "RETFound weights found", _ok,
    f"{WEIGHTS_PATH}",
    "NOT FOUND anywhere under /kaggle/input. Attach the dataset containing "
    "'RETFound_mae_meh.pth' via Add Data.",
)

# 2. Project code found
_ok = _train_py_hit is not None and Path(CODE_ROOT, "src").exists()
_all_ok &= _check(
    "Project code found", _ok,
    f"{CODE_ROOT} (train.py + src/ present)",
    f"train.py and/or src/ not found under {CODE_ROOT}. Attach the "
    "q2l-mmrdr-code dataset via Add Data.",
)

# 3. Data root discovered (drives Config.project_root — see discovery cell)
_ok = DATA_ROOT is not None
_all_ok &= _check(
    "Data root discovered", _ok,
    f"{DATA_ROOT}",
    "Could not locate a directory containing both MMRDR-CFP/FP.csv and "
    "MMRDR-UWF/UWF.csv anywhere under /kaggle/working or /kaggle/input. "
    "Re-run the extraction cell above and check for archive errors.",
)

# 3a. CFP folder
_cfp = (DATA_ROOT / "MMRDR-CFP") if DATA_ROOT else None
_ok = _cfp is not None and _cfp.exists()
_all_ok &= _check(
    "CFP folder found", _ok,
    f"{_cfp}",
    f"{_cfp if _cfp else '<data root not found>'} does not exist.",
)

# 3b. UWF folder
_uwf = (DATA_ROOT / "MMRDR-UWF") if DATA_ROOT else None
_ok = _uwf is not None and _uwf.exists()
_all_ok &= _check(
    "UWF folder found", _ok,
    f"{_uwf}",
    f"{_uwf if _uwf else '<data root not found>'} does not exist.",
)

# 3c. Required CSV files
_fp_csv  = (_cfp / "FP.csv")  if _cfp  else None
_uwf_csv = (_uwf / "UWF.csv") if _uwf  else None
_ok = _fp_csv is not None and _fp_csv.exists() and _uwf_csv is not None and _uwf_csv.exists()
_all_ok &= _check(
    "Annotation CSVs found", _ok,
    f"{_fp_csv} and {_uwf_csv}",
    f"Missing one or both of: {_fp_csv}, {_uwf_csv}.",
)

# 3d. Image directories
_cfp_img = (_cfp / "img") if _cfp else None
_uwf_img = (_uwf / "img") if _uwf else None
_ok = _cfp_img is not None and _cfp_img.exists() and _uwf_img is not None and _uwf_img.exists()
_all_ok &= _check(
    "Image directories found", _ok,
    f"{_cfp_img} and {_uwf_img}",
    f"Missing one or both of: {_cfp_img}, {_uwf_img}. Images may not have "
    "extracted correctly, or this dataset version stores images elsewhere.",
)

# 4. Output directory writable
_ok = True
_detail_fail = ""
try:
    _probe_dir = Path(WORKING_ROOT) / "outputs"
    _probe_dir.mkdir(parents=True, exist_ok=True)
    _probe_file = _probe_dir / ".write_test"
    _probe_file.write_text("ok")
    _probe_file.unlink()
except Exception as e:
    _ok = False
    _detail_fail = f"Cannot write to {WORKING_ROOT}/outputs: {e}"
_all_ok &= _check(
    "Output directory writable", _ok,
    f"{WORKING_ROOT}/outputs",
    _detail_fail,
)

if not _all_ok:
    raise RuntimeError(
        "Pre-flight validation failed — see ✗ items above for exactly "
        "what is missing before training can start."
    )

print("\nAll pre-flight checks passed.")


✓ RETFound weights found: /kaggle/input/datasets/sanchitgarg999/retfound-meh-weights/RETFound_mae_meh.pth
✓ Project code found: /kaggle/input/q2l-mmrdr-code (train.py + src/ present)
✓ Data root discovered: /kaggle/working/dataset
✓ CFP folder found: /kaggle/working/dataset/MMRDR-CFP
✓ UWF folder found: /kaggle/working/dataset/MMRDR-UWF
✓ Annotation CSVs found: /kaggle/working/dataset/MMRDR-CFP/FP.csv and /kaggle/working/dataset/MMRDR-UWF/UWF.csv
✓ Image directories found: /kaggle/working/dataset/MMRDR-CFP/img and /kaggle/working/dataset/MMRDR-UWF/img
✓ Output directory writable: /kaggle/working/outputs

All pre-flight checks passed.


In [9]:
# Order: frozen first (faster, less VRAM) → then fine-tuned
BASELINES = [
    ("B3", "linear", True,  "Frozen RETFound + GAP \u2192 Linear(7)"),
    ("B4", None,     True,  "Frozen RETFound + Q2L Decoder"),
    ("B1", "linear", False, "Fine-tuned RETFound + GAP \u2192 Linear(7)"),
    ("B2", "mlp",    False, "Fine-tuned RETFound + GAP \u2192 MLP(1024\u2192512\u21927)"),
]

print(f"Weights OK: {WEIGHTS_PATH}")
print(f"{len(BASELINES)} baselines queued: {[b[0] for b in BASELINES]}")

Weights OK: /kaggle/input/datasets/sanchitgarg999/retfound-meh-weights/RETFound_mae_meh.pth
4 baselines queued: ['B3', 'B4', 'B1', 'B2']


In [10]:
total_timer = Timer()
completed = []

for bid, head, frozen, desc in BASELINES:
    try:
        run_baseline(bid, head, frozen, desc)
        completed.append(bid)
    except Exception as e:
        print(f"\n\u26a0 Baseline {bid} failed: {e}")
        import traceback
        traceback.print_exc()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f"\nCompleted baselines: {', '.join(completed) or 'none'}")
print(f"Total wall time: {total_timer.elapsed_str()}")

BASELINE B3: Frozen RETFound + GAP → Linear(7)
Output: /kaggle/working/dataset/outputs/baseline_B3
Train: 486 batches | Val: 66 batches
/kaggle/input/q2l-mmrdr-code/src/backbone.py:222: RuntimeWarning: torch.load(weights_only=True) failed on '/kaggle/input/datasets/sanchitgarg999/retfound-meh-weights/RETFound_mae_meh.pth' (UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL argparse.Namespace was not an al

RETFound-MEH checkpoint load report
  Checkpoint path      : /kaggle/input/datasets/sanchitgarg999/retfound-meh-weights/RETFound_mae_meh.pth
  Checkpoint filename  : RETFound_mae_meh.pth
  Checkpoint keys      : 398
  Target model keys    : 294
  Matched (loaded)     : 294/294 (pos_embed interpolated 14x14->32x32)
  Missing (NOT loaded) : 0  (none — every backbone parameter received pretrained weights)
  Unexpected, expected (MAE decoder/mask_token, safely ignored): 104
  Unexpected, UNRECOGNIZED (potential real mismatch)          : 0
  ✅ RETFound-MEH checkpoint loaded successfully — 294/294 backbone parameters initialized from pretrained weights. Model is NOT training from random initialization.


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()
Backbone: 304,149,504 (2,048 trainable)



⚠ Baseline B3 failed: train_one_epoch() got an unexpected keyword argument 'scheduler'


/kaggle/input/q2l-mmrdr-code/src/backbone.py:222: RuntimeWarning: torch.load(weights_only=True) failed on '/kaggle/input/datasets/sanchitgarg999/retfound-meh-weights/RETFound_mae_meh.pth' (UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL argparse.Namespace was not an allowed global by default. Please use `torch.serialization.add_safe_globals([argparse.Namespace])` or the `torch.serialization.safe_global

...


⚠ Baseline B1 failed: train_one_epoch() got an unexpected keyword argument 'scheduler'


/kaggle/input/q2l-mmrdr-code/src/backbone.py:222: RuntimeWarning: torch.load(weights_only=True) failed on '/kaggle/input/datasets/sanchitgarg999/retfound-meh-weights/RETFound_mae_meh.pth' (UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL argparse.Namespace was not an allowed global by default. Please use `torch.serialization.add_safe_globals([argparse.Namespace])` or the `torch.serialization.safe_global

RETFound-MEH checkpoint load report
  Checkpoint path      : /kaggle/input/datasets/sanchitgarg999/retfound-meh-weights/RETFound_mae_meh.pth
  Checkpoint filename  : RETFound_mae_meh.pth
  Checkpoint keys      : 398
  Target model keys    : 294
  Matched (loaded)     : 294/294 (pos_embed interpolated 14x14->32x32)
  Missing (NOT loaded) : 0  (none — every backbone parameter received pretrained weights)
  Unexpected, expected (MAE decoder/mask_token, safely ignored): 104
  Unexpected, UNRECOGNIZED (potential real mismatch)          : 0
  ✅ RETFound-MEH checkpoint loaded successfully — 294/294 backbone parameters initialized from pretrained weights. Model is NOT training from random initialization.


Backbone: 304,149,504 (304,149,504 trainable)
Head:     528,391 (528,391 trainable)
Total:    304,677,895 trainable / 304,677,895

Training (max 75 ep, patience 15)…
Traceback (most recent call last):
  File "/tmp/ipykernel_24/4085864857.py", line 6, in <cell line: 0>
    run_baseline(bid, head, frozen, desc)
  File "/tmp/ipykernel_24/2678989541.py", line 84, in run_baseline
    train_loss, global_step = train_one_epoch(
                              ^^^^^^^^^^^^^^^^
TypeError: train_one_epoch() got an unexpected keyword argument 'scheduler'



⚠ Baseline B2 failed: train_one_epoch() got an unexpected keyword argument 'scheduler'

Completed baselines: none
Total wall time: 53s


In [11]:
print_comparison()


BASELINE COMPARISON — Macro F1 on Test Sets
Experiment                CFP @0.5  CFP @opt  UWF @0.5  UWF @opt
------------------------------------------------------------------------------
B3 (Frozen+Linear)        (results not available)
B4 (Frozen+Q2L)           (results not available)
B1 (FT+Linear)            (results not available)
B2 (FT+MLP)               (results not available)
Main (FT+Q2L)             (results not available)

